# Cleanup and Re-evaluation

Two cleanup steps motivated by the leakage diagnostic findings:

1. **Remove near-duplicate test examples** (max train-similarity ≥ 0.90). Affects ~1% of the test set.
2. **Augment TF-IDF stopwords with format-artifact tokens**. The leakage diagnostic found that TF-IDF's top features included instruction-format verbs (`describe`, `explain`, etc.) and MCQ-stem function words (`following`, `which`) — provenance-detection rather than content. Adding these to the stopword list forces the lexical model to discriminate on content.

All three models are retrained on the cleaned data, with the same evaluation suite as the modeling notebook. Comparing pre- and post-cleanup metrics tells us how much performance was provenance-driven vs content-driven.

Stopwording applies to TF-IDF only. Sentence-Transformer and DistilBERT tokenize the full text and don't honor an external stopword list, so their changes between pre- and post-cleanup come from the near-duplicate test removal alone.

**Runtime**: ~15–20 minutes total on Colab T4 GPU.

## Environment setup

In [ ]:
!pip install -q datasets sentence-transformers scikit-learn pandas pyarrow transformers accelerate matplotlib

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
DATA_DIR = Path("/content/drive/MyDrive/biology_refusal/data")
OUT_DIR  = Path("/content/drive/MyDrive/biology_refusal/models")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

In [ ]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_curve,
    average_precision_score, roc_auc_score, f1_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

SEED = 42
np.random.seed(SEED)

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
test_df  = pd.read_parquet(DATA_DIR / "test.parquet")
print(f"Pre-cleanup — Train: {len(train_df)}, Test: {len(test_df)}")

## Fix 1 — Drop near-duplicate test examples

In [ ]:
# Compute max train-similarity per test example, drop anything >= 0.90.

print("\nFix 1: Removing near-duplicate test examples (max train-sim >= 0.90)...")
encoder = SentenceTransformer("all-MiniLM-L6-v2")

train_emb = encoder.encode(
    train_df["text"].tolist(), batch_size=64,
    show_progress_bar=False, convert_to_numpy=True,
)
test_emb  = encoder.encode(
    test_df["text"].tolist(), batch_size=64,
    show_progress_bar=False, convert_to_numpy=True,
)

sim_matrix = cosine_similarity(test_emb, train_emb)
max_sim_per_test = sim_matrix.max(axis=1)

NEAR_DUP_THR = 0.90
near_dup_mask = max_sim_per_test >= NEAR_DUP_THR
print(f"  Test examples flagged as near-duplicates: {near_dup_mask.sum()}")
print(f"  Per-bucket breakdown of dropped examples:")
print(test_df.loc[near_dup_mask, "source"].value_counts().to_string())

test_df_clean = test_df.loc[~near_dup_mask].reset_index(drop=True)
print(f"\n  Test after cleanup: {len(test_df_clean)} (was {len(test_df)})")
print(f"  Label balance: {Counter(test_df_clean['label'])}")

## Fix 2 — Build the augmented stopword list

In [ ]:
# Stopword the format/imperative artifacts visible in Check 5 of the leakage
# notebook. We keep sklearn's default English stopwords and ADD the format
# tokens that showed up dominating the discriminating features.
#
# Important: this only affects TF-IDF. Sentence-transformers and DistilBERT
# tokenize and process the full text, so adding to a "stopword list" doesn't
# apply to them. For those models we either (a) accept that they may also
# exploit format artifacts to some degree, or (b) preprocess the text to
# strip these tokens before tokenization. We're doing (a) for cost reasons —
# (b) would require re-tokenizing and is more invasive.

# Imperative verbs that dominated the "don't refuse" side (Alpaca format).
imperative_artifacts = {
    "describe", "explain", "generate", "create", "write", "name", "list",
    "suggest", "compose", "tell", "give", "provide", "summarize", "translate",
    "convert", "make", "design", "develop", "compare", "identify", "find",
    "show", "construct", "build", "produce", "draft", "draw", "edit",
}

# Question-stem function words that dominated the "refuse" side (MCQ format).
# We're more conservative here — these words also appear in benign questions,
# so removing them too aggressively kills useful signal. We include only the
# ones that are clearly format-related ("following", "which", etc.).
mcq_artifacts = {
    "following", "which", "would", "could", "best", "true", "correct",
    "incorrect", "above", "below", "among",
}

# Format-counting tokens that show up in instruction formats ("name three...").
counting_artifacts = {
    "three", "two", "five", "four", "ten", "one", "first", "second", "third",
}

artifact_stopwords = (
    set(ENGLISH_STOP_WORDS) | imperative_artifacts | mcq_artifacts | counting_artifacts
)
print(f"\nFix 2: Augmented stopword list has {len(artifact_stopwords)} terms")
print(f"  Added: {len(artifact_stopwords) - len(ENGLISH_STOP_WORDS)} new format/artifact stopwords")

## Shared evaluation function (same as modeling notebook)

In [ ]:
def evaluate(name, y_true, y_pred, y_score, test_df_eval):
    print(f"\n{'='*70}\n  {name}\n{'='*70}")
    print("\nPer-class report:")
    print(classification_report(
        y_true, y_pred,
        target_names=["don't refuse (0)", "refuse (1)"], digits=3,
    ))
    cm = confusion_matrix(y_true, y_pred)
    print(f"Confusion matrix:")
    print(f"           pred 0  pred 1")
    print(f"  true 0:  {cm[0,0]:6d}  {cm[0,1]:6d}")
    print(f"  true 1:  {cm[1,0]:6d}  {cm[1,1]:6d}")
    ap  = average_precision_score(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    print(f"\nAP: {ap:.4f}  ROC-AUC: {auc:.4f}")

    print("\nPer-bucket recall on negatives:")
    df_eval = test_df_eval.copy()
    df_eval["pred"] = y_pred
    for bucket in sorted(df_eval[df_eval["label"] == 0]["source"].unique()):
        sub = df_eval[df_eval["source"] == bucket]
        recall = (sub["pred"] == 0).mean()
        print(f"  {bucket:25s}  n={len(sub):4d}  recall={recall:.3f}")

    print("\nPer-source recall on positives:")
    for source in sorted(df_eval[df_eval["label"] == 1]["source"].unique()):
        sub = df_eval[df_eval["source"] == source]
        recall = (sub["pred"] == 1).mean()
        print(f"  {source:25s}  n={len(sub):4d}  recall={recall:.3f}")

    # Operating points at fixed FPR
    print("\nOperating points (FPR-constrained):")
    fpr_at_thr = []
    for thr in np.sort(np.unique(y_score))[::-1][:300]:
        preds = (y_score >= thr).astype(int)
        tn = ((preds == 0) & (y_true == 0)).sum()
        fp = ((preds == 1) & (y_true == 0)).sum()
        tp = ((preds == 1) & (y_true == 1)).sum()
        fn = ((preds == 0) & (y_true == 1)).sum()
        if tn + fp == 0:
            continue
        fpr = fp / (fp + tn)
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr_at_thr.append((thr, fpr, rec))
    for target_fpr in [0.01, 0.05, 0.10]:
        valid = [(thr, fpr, rec) for thr, fpr, rec in fpr_at_thr if fpr <= target_fpr]
        if valid:
            best = max(valid, key=lambda x: x[2])
            print(f"  FPR<={target_fpr:.2f}: thr={best[0]:.3f}, recall={best[2]:.3f}")
        else:
            print(f"  FPR<={target_fpr:.2f}: no threshold achieves this")
    return {"name": name, "ap": ap, "auc": auc, "y_score": y_score, "y_pred": y_pred}

## Retrain Model 1 — TF-IDF + LogReg with augmented stopwords

In [ ]:
print("\nRetraining Model 1: TF-IDF + LogReg (with format stopwords)...")
y_train = train_df["label"].to_numpy()
y_test  = test_df_clean["label"].to_numpy()

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2, max_df=0.95,
    sublinear_tf=True,
    max_features=50000,
    stop_words=list(artifact_stopwords),
)
X_train_tfidf = tfidf.fit_transform(train_df["text"])
X_test_tfidf  = tfidf.transform(test_df_clean["text"])
print(f"  TF-IDF feature dim after stopwording: {X_train_tfidf.shape[1]}")

tfidf_lr = LogisticRegressionCV(
    Cs=10, cv=5, max_iter=2000, class_weight="balanced",
    scoring="f1", random_state=SEED, n_jobs=-1,
)
tfidf_lr.fit(X_train_tfidf, y_train)

tfidf_pred  = tfidf_lr.predict(X_test_tfidf)
tfidf_score = tfidf_lr.predict_proba(X_test_tfidf)[:, 1]
tfidf_result = evaluate("Model 1: TF-IDF + LogReg (cleaned)", y_test, tfidf_pred, tfidf_score, test_df_clean)

## Retrain Model 2 — Sentence-Transformer + LogReg

In [ ]:
# Embeddings are recomputed against the cleaned test set. Train embeddings
# stay the same.

print("\nRetraining Model 2: Sentence-Transformer + LogReg...")
X_train_emb = train_emb   # reuse from Cell 2
X_test_emb_clean = encoder.encode(
    test_df_clean["text"].tolist(), batch_size=64,
    show_progress_bar=False, convert_to_numpy=True,
)

st_lr = LogisticRegressionCV(
    Cs=10, cv=5, max_iter=2000, class_weight="balanced",
    scoring="f1", random_state=SEED, n_jobs=-1,
)
st_lr.fit(X_train_emb, y_train)
st_pred  = st_lr.predict(X_test_emb_clean)
st_score = st_lr.predict_proba(X_test_emb_clean)[:, 1]
st_result = evaluate("Model 2: Sentence-Transformer + LogReg (cleaned)",
                     y_test, st_pred, st_score, test_df_clean)

## Retrain Model 3 — DistilBERT

In [ ]:
# Same training procedure as the modeling notebook, evaluated on cleaned test.

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nRetraining Model 3: DistilBERT on {device}...")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

train_ds = TextDataset(train_df["text"].tolist(), y_train, tokenizer)
test_ds  = TextDataset(test_df_clean["text"].tolist(), y_test, tokenizer)

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
class_weights = torch.tensor(
    [len(y_train) / (2 * n_neg), len(y_train) / (2 * n_pos)],
    dtype=torch.float,
).to(device)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=class_weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "distilbert_clean"),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
    logging_steps=50, save_strategy="no", eval_strategy="no",
    seed=SEED, report_to="none", fp16=(device == "cuda"),
)
trainer = WeightedTrainer(model=model, args=training_args, train_dataset=train_ds)
trainer.train()

# Evaluate
model.eval()
all_logits = []
batch_size = 32
with torch.no_grad():
    for i in range(0, len(test_ds), batch_size):
        batch = [test_ds[j] for j in range(i, min(i + batch_size, len(test_ds)))]
        input_ids      = torch.stack([b["input_ids"] for b in batch]).to(device)
        attention_mask = torch.stack([b["attention_mask"] for b in batch]).to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        all_logits.append(logits.cpu().numpy())
all_logits = np.concatenate(all_logits, axis=0)
exp = np.exp(all_logits - all_logits.max(axis=1, keepdims=True))
probs = exp / exp.sum(axis=1, keepdims=True)
db_score = probs[:, 1]
db_pred  = (db_score >= 0.5).astype(int)
db_result = evaluate("Model 3: DistilBERT (cleaned)", y_test, db_pred, db_score, test_df_clean)

## Side-by-side comparison + delta from pre-cleanup

In [ ]:
print("\n" + "=" * 70)
print("  CLEANED-DATA HEADLINE COMPARISON")
print("=" * 70)
results = [tfidf_result, st_result, db_result]
summary = pd.DataFrame([
    {
        "model": r["name"].split("(")[0].strip(),
        "F1 (pos)": f1_score(y_test, r["y_pred"], pos_label=1),
        "F1 (neg)": f1_score(y_test, r["y_pred"], pos_label=0),
        "AP":       r["ap"],
        "ROC-AUC":  r["auc"],
    }
    for r in results
])
print(summary.to_string(index=False))

# PR curves overlay
fig, ax = plt.subplots(figsize=(7, 5))
for r in results:
    p, rec, _ = precision_recall_curve(y_test, r["y_score"])
    ax.plot(rec, p, label=f"{r['name'].split(':')[0]} (AP={r['ap']:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("PR curves on CLEANED test set")
ax.legend(loc="lower left", fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "pr_curves_clean.png", dpi=120)
plt.show()

# Per-bucket comparison
print("\n" + "=" * 70)
print("  PER-BUCKET RECALL ON NEGATIVES (cleaned)")
print("=" * 70)
bucket_table = []
for r in results:
    row = {"model": r["name"].split(":")[0]}
    df_eval = test_df_clean.copy()
    df_eval["pred"] = r["y_pred"]
    for bucket in sorted(df_eval[df_eval["label"] == 0]["source"].unique()):
        sub = df_eval[df_eval["source"] == bucket]
        row[bucket] = (sub["pred"] == 0).mean()
    bucket_table.append(row)
print(pd.DataFrame(bucket_table).to_string(index=False))

## Error analysis dumps (cleaned data)

In [ ]:
err_dir = OUT_DIR / "errors_clean"
err_dir.mkdir(exist_ok=True)
for r in results:
    name = r["name"].split(":")[0].lower().replace(" ", "_")
    df_err = test_df_clean.copy()
    df_err["pred"]  = r["y_pred"]
    df_err["score"] = r["y_score"]
    fp = df_err[(df_err["label"] == 0) & (df_err["pred"] == 1)].sort_values("score", ascending=False)
    fn = df_err[(df_err["label"] == 1) & (df_err["pred"] == 0)].sort_values("score", ascending=True)
    fp.to_csv(err_dir / f"{name}_false_positives.csv", index=False)
    fn.to_csv(err_dir / f"{name}_false_negatives.csv", index=False)
    print(f"  {name}: {len(fp)} FPs, {len(fn)} FNs")

preds_df = test_df_clean.copy()
for r in results:
    name = r["name"].split(":")[0].lower().replace(" ", "_")
    preds_df[f"{name}_pred"]  = r["y_pred"]
    preds_df[f"{name}_score"] = r["y_score"]
preds_df.to_parquet(OUT_DIR / "all_predictions_clean.parquet", index=False)
print(f"\nSaved predictions and error CSVs to {OUT_DIR}/")

## Inspect features after stopwording

In [ ]:
# Sanity check that the format artifacts are gone from the top features.

from sklearn.linear_model import LogisticRegression
vec = TfidfVectorizer(
    ngram_range=(1, 1), min_df=3, max_df=0.95, max_features=20000,
    stop_words=list(artifact_stopwords),
)
X_tr = vec.fit_transform(train_df["text"])
lr = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
lr.fit(X_tr, y_train)
feature_names = np.array(vec.get_feature_names_out())
coefs = lr.coef_[0]

print("\nTop 20 REFUSE features AFTER stopwording (sanity check):")
top_pos = np.argsort(-coefs)[:20]
for i in top_pos:
    print(f"  {feature_names[i]:30s}  coef={coefs[i]:+.3f}")
print("\nTop 20 DON'T REFUSE features AFTER stopwording:")
top_neg = np.argsort(coefs)[:20]
for i in top_neg:
    print(f"  {feature_names[i]:30s}  coef={coefs[i]:+.3f}")